<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)

Mounted at /content/drive
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [2]:
CLEANED_FILES = [
    "train_s1_cleaned.tsv",
    "train_s2_cleaned.tsv",
    "train_s3_cleaned.tsv",
    "test_s1_cleaned.tsv",
    "test_s2_cleaned.tsv",
    "test_s3_cleaned.tsv"
]

print("Checking cleaned files...\n")

for filename in CLEANED_FILES:

    path = os.path.join(
        CLEANED_DATA_ROOT,
        filename
    )

    print(
        f"{filename:<25}",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

Checking cleaned files...

train_s1_cleaned.tsv      EXISTS
train_s2_cleaned.tsv      EXISTS
train_s3_cleaned.tsv      EXISTS
test_s1_cleaned.tsv       EXISTS
test_s2_cleaned.tsv       EXISTS
test_s3_cleaned.tsv       EXISTS


In [3]:
sample_path = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

sample = pd.read_csv(
    sample_path,
    sep="\t",
    nrows=5
)

print("Columns:")
print(sample.columns.tolist())

print("\nSample:")
display(sample)

Columns:
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_missing', 'address_missing', 'name_token_count', 'address_token_count', 'name_core']

Sample:


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_missing,address_missing,name_token_count,address_token_count,name_core
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester dr high point nc,us,0,0,3,6,orelee s barbershop
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis rd tahlequah ok,us,0,0,2,5,prime money
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello ave phoenix az,us,0,0,3,5,b retail
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron dr unit apt g dundalk md,us,0,0,2,8,christ chapel
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,india,0,0,3,9,prabhav business center


Blocker 1 (country_norm + name_core)

In [4]:
import pandas as pd
from collections import Counter

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv"
)

CHUNK_SIZE = 200_000

def get_block_sizes(path):
  counter = Counter()

  for chunk in pd.read_csv(
      path,
      sep="\t",
      chunksize=CHUNK_SIZE,
      dtype={"country_norm": "string", "name_norm": "string"}):
      chunk = chunk[
          chunk["name_core"].notna() &
          (chunk["name_core"] != "")
      ]

      keys = zip(
          chunk["country_norm"],
          chunk["name_core"]
      )

      counter.update(keys)

      del chunk
      gc.collect()

  return counter

print("Building S2 block statistics...")
s2_blocks = get_block_sizes(S2_PATH)

print("Building S3 block statistics...")
s3_blocks = get_block_sizes(S3_PATH)

print("\nS2 unique blocks:", len(s2_blocks))
print("S3 unique blocks:", len(s3_blocks))


Building S2 block statistics...
Building S3 block statistics...

S2 unique blocks: 3597772
S3 unique blocks: 3850257


In [5]:
print("\nLargest S2 blocks:")
for key, count in s2_blocks.most_common(20):
    print(count, "->", key)

print("\nLargest S3 blocks:")
for key, count in s3_blocks.most_common(20):
    print(count, "->", key)


Largest S2 blocks:
647 -> ('us', 'meridian')
563 -> ('us', 'physical therapy')
550 -> ('us', 'primary care')
514 -> ('us', 'womens health')
507 -> ('us', 'internal medicine')
500 -> ('us', 'behavioral health')
494 -> ('us', 'urgent care')
491 -> ('us', 'pediatric dental')
482 -> ('us', 'pediatric dentistry')
408 -> ('us', 'ear nose throat')
398 -> ('us', 'foot ankle')
375 -> ('us', 'family center')
374 -> ('us', 'summit')
374 -> ('us', 'lynx')
366 -> ('us', 'anchor')
362 -> ('us', 'sapphire')
360 -> ('us', 'helios')
359 -> ('us', 'falcon')
356 -> ('us', 'cedar')
355 -> ('us', 'earnosethroat com')

Largest S3 blocks:
584 -> ('us', 'meridian')
548 -> ('us', 'primary care')
538 -> ('us', 'physical therapy')
524 -> ('us', 'pediatric dental')
507 -> ('us', 'urgent care')
492 -> ('us', 'womens health')
490 -> ('us', 'pediatric dentistry')
458 -> ('us', 'behavioral health')
452 -> ('us', 'internal medicine')
426 -> ('us', 'summit')
418 -> ('us', 'cascade')
412 -> ('us', 'family center')
399 

In [6]:
from collections import defaultdict
import gc

def build_block_index(path):

    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):

        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [7]:
print("Building S2 index...")
s2_index = build_block_index(S2_PATH)
print("S2 index blocks:", len(s2_index))
gc.collect()
print("\nBuilding S3 index...")
s3_index = build_block_index(S3_PATH)
print("S3 index blocks:", len(s3_index))

Building S2 index...
S2 index blocks: 3597772

Building S3 index...
S3 index blocks: 3850257


In [8]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
)

print("S1 rows:", len(s1))
print("Ground truth rows:", len(gt))

S1 rows: 2206821
Ground truth rows: 2206821


In [12]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))

Ground truth lookup entries: 2206821


In [11]:
def calculate_block_recall_fast(s1_df, gt_lookup, index, prefix):

    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        candidates = index.get((country, name), [])

        candidate_set = set(candidates)

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches
        else 0
    )

    return total_matches, captured_matches, recall

In [13]:
print("Evaluating S2 blocking recall...")

s2_total, s2_captured, s2_recall = calculate_block_recall_fast(
    s1,
    gt_lookup,
    s2_index,
    "S2-"
)

print("\nS2")
print("Total true matches :", s2_total)
print("Captured matches   :", s2_captured)
print(f"Blocking recall    : {s2_recall:.4%}")

Evaluating S2 blocking recall...

S2
Total true matches : 3693619
Captured matches   : 1447011
Blocking recall    : 39.1760%


In [16]:
print("Evaluating S3 blocking recall...")

s3_total, s3_captured, s3_recall = calculate_block_recall_fast(
    s1,
    gt_lookup,
    s3_index,
    "S3-"
)

print("\nS3")
print("Total true matches :", s3_total)
print("Captured matches   :", s3_captured)
print(f"Blocking recall    : {s3_recall:.4%}")

Evaluating S3 blocking recall...

S3
Total true matches : 3944746
Captured matches   : 1490930
Blocking recall    : 37.7953%
